In [1]:
print("test")

test


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from utils.data_processing import load_data, standarize, variance_threshold, correlation_filter, select_from_model

In [2]:
X, Y, XfinalTest = load_data()
X, XfinalTest = standarize(X, XfinalTest)
X, XfinalTest = variance_threshold(X, XfinalTest, 0.01)
X, XfinalTest = correlation_filter(X, XfinalTest, 0.9)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

def custom_score(estimator, X, y):

    y_pred = estimator.predict(X)
    y = np.array(y).ravel()
    
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))

    used_features = np.unique(
        estimator.tree_.feature[
            estimator.tree_.feature >= 0
        ]
    )

    no_variables = len(used_features)

    return (tp * 10) - (fp * 5) - (no_variables * 200)

grid = GridSearchCV(
    DecisionTreeClassifier(),
    {
        "max_depth": [2, 3, 5, 10],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },
    scoring=custom_score,
    cv=5
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

Best params: {'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best score: 212.0


In [17]:
X.shape

(5000, 493)

In [14]:
X, XfinalTest = select_from_model(X, Y, XfinalTest)

c:\Users\tomas\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


ValueError: Item wrong length 493 instead of 5000.

In [43]:
def xgb_custom_score(estimator, X, y):

    y_pred = estimator.predict(X)
    y = np.array(y).ravel()
    
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))

    used_features = np.unique(
        estimator.get_booster().get_dump()[0].split("f")[1:]
    )

    no_variables = len(used_features)

    return (tp * 10) - (fp * 5) - (no_variables * 200)

In [56]:
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
# fit model
bst.fit(X_train, y_train)
# make predictions
preds = bst.predict(X_test)

grid = GridSearchCV(
    XGBClassifier(),
    {
        "n_estimators": [1, 2, 3, 5, 10],
        "max_depth": [1, 2, 3, 5, 10],
        "learning_rate": [0.1, 0.5, 1.0]
    },
    scoring=xgb_custom_score,
    cv=5
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

Best params: {'learning_rate': 1.0, 'max_depth': 1, 'n_estimators': 5}
Best score: 1150.0


In [57]:
xgb_custom_score(grid.best_estimator_, X_test, y_test)

np.int64(1505)